# Extreme Learning Machine (ELM)

## Importing the libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

## Importing the dataset

In [43]:
df = pd.read_csv("./Dataset/Concrete_Strength.csv")

def strength_class(x):
    if x < 20:
        return 0  # Low
    elif x < 40:
        return 1  # Medium
    else:
        return 2  # High

df["Strength_Class"] = df["Concrete_compressive_strength"].apply(strength_class)

# Features & target
X = df.drop(columns=["Concrete_compressive_strength", "Strength_Class"]).fillna(0.0)
y = df["Strength_Class"]

## Splitting the dataset into the Training set and Test set

In [44]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Feature Scaling

In [45]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Training the ELM model on the Training set

In [23]:
class ELMClassifier:
    def __init__(self, input_dim, hidden_dim=500, activation='relu'):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.activation = activation
        self.W = np.random.randn(input_dim, hidden_dim)
        self.b = np.random.randn(hidden_dim)

    def _activation(self, X):
        if self.activation == 'sigmoid':
            return 1 / (1 + np.exp(-X))
        elif self.activation == 'relu':
            return np.maximum(0, X)
        else:
            return X

    def fit(self, X, y):
        y_onehot = np.zeros((y.size, y.max() + 1))
        y_onehot[np.arange(y.size), y] = 1
        H = self._activation(np.dot(X, self.W) + self.b)
        self.beta = np.linalg.pinv(H) @ y_onehot

    def predict(self, X):
        H = self._activation(np.dot(X, self.W) + self.b)
        y_pred_prob = H @ self.beta
        return np.argmax(y_pred_prob, axis=1)


elm_clf = ELMClassifier(input_dim=X_train_scaled.shape[1], hidden_dim=500, activation='relu')
elm_clf.fit(X_train_scaled, y_train)

## Predicting the Test set results

In [24]:
y_pred = elm_clf.predict(X_test_scaled)

print("\n📊 ELM Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Low", "Medium", "High"]))



📊 ELM Classification Report:
              precision    recall  f1-score   support

         Low       0.73      0.82      0.77        39
      Medium       0.78      0.81      0.80        91
        High       0.87      0.76      0.81        76

    accuracy                           0.80       206
   macro avg       0.79      0.80      0.79       206
weighted avg       0.80      0.80      0.80       206



## Making the Confusion Matrix

In [25]:
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

Confusion Matrix:
 [[32  7  0]
 [ 8 74  9]
 [ 4 14 58]]
Accuracy: 79.61%


# ADMM Top 5 features(SVM)

## ADMM Feature Selection

In [26]:
def select_top_k_via_admm(X, y, lam=0.01, rho=1.0, iters=200, k=5):
    Xn = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-12)
    Xn = Xn.values if hasattr(Xn, "values") else np.asarray(Xn)
    yv = y.values if hasattr(y, "values") else np.asarray(y)

    n, p = Xn.shape
    w = np.zeros(p)
    z = np.zeros(p)
    u = np.zeros(p)

    XtX = Xn.T @ Xn
    Xty = Xn.T @ yv
    I = np.eye(p)

    def soft_threshold(v, t):
        return np.sign(v) * np.maximum(np.abs(v) - t, 0.0)

    for _ in range(iters):
        w = np.linalg.solve(XtX + rho * I, Xty + rho * (z - u))
        z = soft_threshold(w + u, lam / rho)
        u = u + (w - z)

    idx = np.argsort(-np.abs(z))[:k]
    selected_cols = list(X.columns[idx])
    return selected_cols

top5_cols = select_top_k_via_admm(X_train, y_train, lam=0.01, rho=1.0, iters=300, k=5)
print("Selected (Top-5 ADMM):", top5_cols)

X = X[top5_cols]

Selected (Top-5 ADMM): ['Cement (component 1)(kg in a m^3 mixture)', 'Age (day)', 'Blast Furnace Slag (component 2)(kg in a m^3 mixture)', 'Fly Ash (component 3)(kg in a m^3 mixture)', 'Water  (component 4)(kg in a m^3 mixture)']


## Splitting the dataset into the Training set and Test set

In [27]:
# Full feature matrix (no target columns)
X_all = X
y_cls_all = df["Strength_Class"].to_numpy()
y_reg_all = df["Concrete_compressive_strength"].to_numpy()  # for feature selection (regression)


X_train, X_test, y_train_cls, y_test_cls, y_train_reg, y_test_reg = train_test_split(X_all, y_cls_all, y_reg_all, test_size=0.2, random_state=42, stratify=y_cls_all)


## Feature Scaling

In [28]:
clf_scaler = StandardScaler()
X_train_clf = clf_scaler.fit_transform(X_train)
X_test_clf  = clf_scaler.transform(X_test)

## Training the ELM model on the Training set

In [29]:
elm = ELMClassifier(input_dim=X_train_clf.shape[1], hidden_dim=500, activation='relu')
elm.fit(X_train_clf, y_train_cls)

y_pred = elm.predict(X_test_clf)

## Predicting the Test set results

In [30]:
print("\n📊 ELM Classification Report:")
print(classification_report(y_test_cls, y_pred, target_names=["Low", "Medium", "High"]))



📊 ELM Classification Report:
              precision    recall  f1-score   support

         Low       0.71      0.87      0.78        39
      Medium       0.73      0.73      0.73        91
        High       0.72      0.64      0.68        76

    accuracy                           0.72       206
   macro avg       0.72      0.75      0.73       206
weighted avg       0.72      0.72      0.72       206



## Making the Confusion Matrix

In [31]:
print("Confusion Matrix:\n", confusion_matrix(y_test_cls, y_pred))
print(f"Accuracy: {accuracy_score(y_test_cls, y_pred) * 100:.2f}%")

Confusion Matrix:
 [[34  4  1]
 [ 7 66 18]
 [ 7 20 49]]
Accuracy: 72.33%


## Online Learing in ELM

In [34]:
from time import perf_counter
RANDOM_STATE = 42
TOP_K = 5                 # number of features to keep via ADMM (Lasso)
HIDDEN_DIM = 300          # ELM hidden neurons
ACTIVATION = 'relu'       # 'relu' or 'sigmoid'
RIDGE = 1e-2              # RLS regularization (lambda)
PRINT_EVERY = 200         # progress print frequency during online updates

rng = np.random.default_rng(RANDOM_STATE)

In [ ]:
class OnlineELMClassifier:

    def __init__(self, input_dim, n_classes, hidden_dim=300, activation='relu', ridge=1e-2, rng=None):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.n_classes = n_classes
        self.activation = activation
        self.ridge = ridge
        self.rng = np.random.default_rng() if rng is None else rng

        if activation == 'relu':
            self.W = self.rng.normal(0, np.sqrt(2.0 / input_dim), size=(input_dim, hidden_dim))
        else:
            self.W = self.rng.normal(0, 1.0 / np.sqrt(input_dim), size=(input_dim, hidden_dim))
        self.b = self.rng.normal(0, 1e-3, size=(hidden_dim,))

        
        self.beta = np.zeros((hidden_dim, n_classes)) # Output weights beta (hidden_dim x n_classes)
        
        self.P = (1.0 / self.ridge) * np.eye(hidden_dim) # RLS inverse correlation matrix P (hidden_dim x hidden_dim)

    def _act(self, Z):
        if self.activation == 'relu':
            return np.maximum(0.0, Z)
        elif self.activation == 'sigmoid':
            return 1.0 / (1.0 + np.exp(-Z))
        else:
            return Z  # linear

    def _hidden(self, x_row):
        return self._act(x_row @ self.W + self.b)  

    def partial_fit(self, x_row, y_label):
        """
        Update beta and P with a single sample.
        y_label is an integer in [0, n_classes-1].
        """
        h = self._hidden(x_row)              
        h = h.reshape(-1, 1)    
        
        y = np.zeros((self.n_classes, 1))
        y[y_label, 0] = 1.0

        # RLS gain and updates
        # k = P h / (1 + h^T P h)
        Ph = self.P @ h                    
        denom = 1.0 + (h.T @ Ph)[0, 0]
        k = Ph / denom                     

        y_pred = (h.T @ self.beta).T        
        e = y - y_pred                      

        self.beta += k @ e.T                
        self.P -= k @ (h.T @ self.P)      

    def predict(self, X):
        H = self._act(X @ self.W + self.b)   
        scores = H @ self.beta               
        return np.argmax(scores, axis=1)

n_classes = len(np.unique(y_train_cls))
clf = OnlineELMClassifier(
    input_dim=X_train_clf.shape[1],
    n_classes=n_classes,
    hidden_dim=HIDDEN_DIM,
    activation=ACTIVATION,
    ridge=RIDGE,
    rng=rng
)

t0 = perf_counter()
for i in range(X_train_clf.shape[0]):
    clf.partial_fit(X_train_clf[i], int(y_train_cls[i]))
    if (i + 1) % PRINT_EVERY == 0:
        print(f"Online updates processed: {i+1}/{X_train_clf.shape[0]}")
train_time = perf_counter() - t0
print(f"\nOnline training time: {train_time:.3f} s")

Online updates processed: 200/824
Online updates processed: 400/824
Online updates processed: 600/824
Online updates processed: 800/824

Online training time: 0.117 s


## Predicting Test Results

In [48]:
y_pred = clf.predict(X_test_clf)

## Analysis

In [49]:
print("\n📊 Online ELM Classification Report (Top-5 ADMM features):")
print(classification_report(y_test_cls, y_pred, target_names=["Low", "Medium", "High"]))
print("Confusion Matrix:\n", confusion_matrix(y_test_cls, y_pred))
print(f"Accuracy: {accuracy_score(y_test_cls, y_pred) * 100:.2f}%")


📊 Online ELM Classification Report (Top-5 ADMM features):
              precision    recall  f1-score   support

         Low       0.79      0.85      0.81        39
      Medium       0.70      0.81      0.75        91
        High       0.84      0.64      0.73        76

    accuracy                           0.76       206
   macro avg       0.78      0.77      0.77       206
weighted avg       0.77      0.76      0.76       206

Confusion Matrix:
 [[33  6  0]
 [ 8 74  9]
 [ 1 26 49]]
Accuracy: 75.73%
